In [ ]:
# Image to Blender Studio
# Enable a Kaggle T4 x2 accelerator and Internet, run this cell, then open the studio link below.
from pathlib import Path
from io import BytesIO
from html import escape
from importlib.metadata import PackageNotFoundError, version
from zipfile import ZipFile, ZIP_DEFLATED
from concurrent.futures import ThreadPoolExecutor, as_completed
from contextlib import ExitStack
import hashlib
import os
import secrets
import re
import shutil
import subprocess
import sys
import time
import traceback
import uuid

import requests
from PIL import Image, ImageOps
from IPython.display import HTML, display

WORK = Path("/kaggle/working")
CACHE = Path("/kaggle/temp/image-to-blender")
CACHE.mkdir(parents=True, exist_ok=True)
COMFY = CACHE / "ComfyUI"
HUNYUAN = CACHE / "Hunyuan3D-2"
OUTPUT = WORK / "blender_assets"
# Set to False if two Qwen workers exceed the session's available system RAM.
PARALLEL_QWEN = True

def gpu_devices():
    if not shutil.which("nvidia-smi"):
        return []
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader"],
        capture_output=True, text=True,
    )
    devices = result.stdout.split() if result.returncode == 0 else []
    visible = os.environ.get("CUDA_VISIBLE_DEVICES")
    if visible is None:
        return devices
    selected = []
    for name in visible.split(","):
        name = name.strip()
        if name.isdecimal() and int(name) < len(devices):
            selected.append(devices[int(name)])
        elif name.startswith("GPU-"):
            selected.extend(device for device in devices if device.startswith(name))
    return list(dict.fromkeys(selected))
PRESETS = {"Draft": (512, 12, 20, 196), "Balanced": (640, 18, 30, 256),
           "Detail": (832, 25, 40, 384)}
WEIGHTS = {
    "diffusion_models": "qwen_image_2.1_int8_convrot.safetensors",
    "text_encoders": "qwen3vl_8b_int8_convrot.safetensors",
    "vae": "qwen_image_2.1_vae_bf16.safetensors",
}
ANGLES = {
    "Front": "an exact straight-on front view",
    "Left": "an exact left profile, 90 degrees around the object from the front",
    "Back": "an exact straight-on rear view, 180 degrees from the front",
    "Right": "an exact right profile, 90 degrees from the front on the opposite side from the left profile",
}

def start_comfy(device, worker, directory):
    port = 8195 + worker
    api = f"http://127.0.0.1:{port}"
    log = directory / f"comfy-{worker}.log"
    with log.open("w") as stream:
        process = subprocess.Popen(
            [sys.executable, "main.py", "--listen", "127.0.0.1",  "--port", str(port),
             "--force-fp16", "--preview-method", "none",
             "--output-directory", str(CACHE / f"comfy-output-{worker}")],
            cwd=COMFY, stdout=stream, stderr=subprocess.STDOUT,
            env=dict(os.environ, CUDA_VISIBLE_DEVICES=device),
        )
    try:
        deadline = time.monotonic() + 180
        while process.poll() is None and time.monotonic() < deadline:
            try:
                response = requests.get(api + "/object_info/TextEncodeQwenImage21", timeout=2)
                if response.ok and "TextEncodeQwenImage21" in response.json():
                    return process, api
            except requests.RequestException:
                pass
            time.sleep(1)
        raise RuntimeError(f"ComfyUI could not load Qwen-Image-2.1. See {log}.")
    except BaseException:
        stop_comfy(process)
        raise

def stop_comfy(process):
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()

def prepare_comfy():
    if not (COMFY / "main.py").is_file():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/Comfy-Org/ComfyUI.git", str(COMFY)], check=True)
    stamp = Path("/tmp/image-to-blender-comfy-ready")
    if not stamp.exists():
        log = COMFY.parent / "image-to-blender-comfy-setup.log"
        with log.open("w") as stream:
            completed = subprocess.run(
                [sys.executable, "-m", "pip", "install", "-r", str(COMFY / "requirements.txt")],
                stdout=stream, stderr=subprocess.STDOUT,
            )
        if completed.returncode:
            raise RuntimeError("ComfyUI setup failed. Check image-to-blender-comfy-setup.log in Kaggle.")
        stamp.touch()
    for folder, name in WEIGHTS.items():
        if not (COMFY / "models" / folder / name).is_file():
            subprocess.run([sys.executable, "-c",
                            "from huggingface_hub import hf_hub_download; import sys; "
                            "hf_hub_download('Comfy-Org/Qwen-Image-2.1', filename=sys.argv[1], local_dir=sys.argv[2])",
                            f"{folder}/{name}", str(COMFY / "models")],
                           env=dict(os.environ, HF_HUB_DISABLE_XET="1"), check=True)

def prepare_reference(image, side, aspect=None):
    image = ImageOps.exif_transpose(image).convert("RGBA")
    ratio = max(0.25, min(4, aspect or image.width / image.height))
    width = max(32, round(side * ratio**0.5 / 32) * 32)
    height = max(32, round(side / ratio**0.5 / 32) * 32)
    image = ImageOps.pad(image, (width, height), Image.Resampling.LANCZOS,
                         color=(255, 255, 255, 0))
    canvas = Image.new("RGB", image.size, "white")
    canvas.paste(image, mask=image.getchannel("A"))
    png = BytesIO()
    canvas.save(png, format="PNG")
    payload = png.getvalue()
    filename = hashlib.sha256(payload).hexdigest() + ".png"
    folder = COMFY / "input" / "asset_refs"
    folder.mkdir(parents=True, exist_ok=True)
    target = folder / filename
    if not target.exists():
        target.write_bytes(payload)
    return f"asset_refs/{filename}"

def view_graph(ref, front, subject, angle, steps, seed):
    def node(kind, **inputs):
        return {"class_type": kind, "inputs": inputs}

    slots = {"images.image_1": ["4", 0]}
    graph = {
        "1": node("UNETLoader", unet_name=WEIGHTS["diffusion_models"], weight_dtype="default"),
        "2": node("CLIPLoader", clip_name=WEIGHTS["text_encoders"], type="qwen_image", device="default"),
        "3": node("VAELoader", vae_name=WEIGHTS["vae"]),
        "4": node("LoadImage", image=ref),
        "7": node("QwenImage21Cache", model=["1", 0], device="auto", dtype="default"),
        "8": node("KSampler", model=["7", 0], positive=["6", 0], negative=["6", 1],
                  latent_image=["6", 2], seed=seed, steps=steps, cfg=1.0,
                  sampler_name="euler", scheduler="simple", denoise=1.0),
        "9": node("VAEDecode", samples=["8", 0], vae=["3", 0]),
        "10": node("SaveImage", images=["9", 0], filename_prefix="asset_views/view"),
    }
    orientation = ""
    if front:
        graph["5"] = node("LoadImage", image=front)
        slots["images.image_2"] = ["5", 0]
        orientation = "The canonical front view is <image2>. Use it to establish orientation. "
    prompt = (
        f"Create {angle} of {subject} in <image1>. {orientation}"
        "Preserve the object's identity, materials, colors, proportions and individual parts. "
        "Keep the object upright and fully visible. Use an eye-level orthographic camera, "
        "center the object on a pure white background, and output one image. "
        "Exclude other objects, people, labels added to the image, shadows and borders."
    )
    graph["6"] = node("TextEncodeQwenImage21", clip=["2", 0], vae=["3", 0], prompt=prompt,
                      negative_prompt="", resolution=0, **slots)
    return graph

def render_view(api, graph, target):
    response = requests.post(api + "/prompt", json={"prompt": graph}, timeout=30)
    if not response.ok:
        raise RuntimeError(f"ComfyUI rejected the view graph: {response.text[:1500]}")
    job = response.json()["prompt_id"]
    deadline = time.monotonic() + 1800
    while time.monotonic() < deadline:
        response = requests.get(f"{api}/history/{job}", timeout=15)
        response.raise_for_status()
        result = response.json().get(job)
        if result:
            images = result.get("outputs", {}).get("10", {}).get("images", [])
            if not images:
                errors = [data.get("exception_message") for kind, data
                          in result.get("status", {}).get("messages", []) if kind == "execution_error"]
                raise RuntimeError(errors[0] if errors and errors[0] else "Qwen returned no image.")
            response = requests.get(api + "/view", params=images[0], timeout=60)
            response.raise_for_status()
            with Image.open(BytesIO(response.content)) as image:
                temporary = target.with_suffix(".tmp")
                image.convert("RGB").save(temporary, format="PNG")
                temporary.replace(target)
            return
        time.sleep(1)
    raise TimeoutError("Qwen did not finish this view within 30 minutes. See the corresponding ComfyUI worker log.")

HUNYUAN_REPO = "tencent/Hunyuan3D-2mv"
HUNYUAN_SUBFOLDER = "hunyuan3d-dit-v2-mv"
HUNYUAN_WEIGHTS = CACHE / "hunyuan-weights"
HUNYUAN_DEPS = CACHE / "hunyuan-transformers-4.57.6"
PREFETCH_SCRIPT = """
from huggingface_hub import hf_hub_download
import sys

repo, subfolder, target = sys.argv[1:]
for filename in ("config.yaml", "model.fp16.safetensors"):
    print(f"Fetching {repo}/{subfolder}/{filename}", flush=True)
    path = hf_hub_download(repo, f"{subfolder}/{filename}", local_dir=target)
    print(f"Ready: {path}", flush=True)
"""

def start_hunyuan_download(directory):
    with (directory / "hunyuan-download.log").open("w") as stream:
        return subprocess.Popen(
            [sys.executable, "-u", "-c", PREFETCH_SCRIPT, HUNYUAN_REPO,
             HUNYUAN_SUBFOLDER, str(HUNYUAN_WEIGHTS)],
            env=dict(os.environ, HF_HOME=str(CACHE / "huggingface")),
            stdout=stream, stderr=subprocess.STDOUT,
        )

def diagnostics(directory):
    files = sorted(path for path in directory.glob("*.log") if path.name != "diagnostics.log")
    text = "\n".join(
        f"=== {path.name} ===\n{path.read_text(encoding='utf-8', errors='replace')}\n"
        for path in files
    )
    report = directory / "diagnostics.log"
    report.write_text(text, encoding="utf-8")
    return str(report)


MESH_SCRIPT = """
from pathlib import Path
import sys
import torch
import transformers
print(f"Transformers: {transformers.__version__}", flush=True)
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline

directory = Path(sys.argv[1])
weights = Path(sys.argv[2])
steps, resolution, seed = map(int, sys.argv[3:])
print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print(f"Loading Hunyuan from {weights}")
pipe = Hunyuan3DDiTFlowMatchingPipeline.from_single_file(
    str(weights / "model.fp16.safetensors"), str(weights / "config.yaml"),
    use_safetensors=True, device="cuda", dtype=torch.float16,
)
print("Hunyuan loaded. Reconstructing mesh.")
views = {angle: str(directory / f"{angle}.png") for angle in ("front", "left", "back", "right")}
with torch.inference_mode():
    mesh = pipe(image=views, generator=torch.Generator(device="cuda").manual_seed(seed),
                num_inference_steps=steps, octree_resolution=resolution, num_chunks=12000,
                output_type="trimesh")[0]
if mesh is None or not len(mesh.faces):
    raise RuntimeError("Hunyuan returned an empty mesh. Try a clearer photo or another seed.")
mesh.export(str(directory / "asset.glb"))
print(f"Exported {len(mesh.faces):,} faces.")
"""

def make_mesh(directory, steps, resolution, seed, device):
    with (directory / "hunyuan-setup.log").open("w") as stream:
        if not (HUNYUAN / "hy3dgen").is_dir():
            subprocess.run(
                ["git", "clone", "--depth", "1",
                 "https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git", str(HUNYUAN)],
                check=True, stdout=stream, stderr=subprocess.STDOUT,
            )
        ready = CACHE / "hunyuan-ready"
        if not ready.exists():
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "omegaconf", "trimesh",
                 "scikit-image", "opencv-python-headless", "einops", "pymeshlab", "diffusers", "accelerate"],
                check=True, stdout=stream, stderr=subprocess.STDOUT,
            )
            ready.touch()

        # Hunyuan loads the checkpoint directly into the Transformers 4.x DINOv2 layout.
        # Install this stack in a private location so ComfyUI keeps its own dependencies.
        dependency_marker = HUNYUAN_DEPS / ".ready"
        if not dependency_marker.exists():
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "--target", str(HUNYUAN_DEPS),
                 "--upgrade", "--no-deps", "transformers==4.57.6", "tokenizers>=0.22,<=0.23.0",
                 "huggingface-hub==0.36.0", "diffusers==0.36.0"],
                check=True, stdout=stream, stderr=subprocess.STDOUT,
            )
            dependency_marker.touch()

    env = dict(os.environ, CUDA_VISIBLE_DEVICES=device, HF_HOME=str(CACHE / "huggingface"),
               PYTHONPATH=os.pathsep.join([str(HUNYUAN_DEPS), str(HUNYUAN), os.environ.get("PYTHONPATH", "")]))
    log = directory / "hunyuan-inference.log"
    with log.open("w") as stream:
        result = subprocess.run(
            [sys.executable, "-u", "-c", MESH_SCRIPT, str(directory),
             str(HUNYUAN_WEIGHTS / HUNYUAN_SUBFOLDER), str(steps), str(resolution), str(seed)],
            env=env, stdout=stream, stderr=subprocess.STDOUT, timeout=3600,
        )
    if result.returncode:
        raise RuntimeError(f"Hunyuan mesh worker exited with code {result.returncode}.")
    with (directory / "asset.glb").open("rb") as asset:
        if asset.read(4) != b"glTF":
            raise RuntimeError("Hunyuan did not produce a valid GLB header.")


try:
    gradio_major = int(version("gradio").split(".")[0])
except PackageNotFoundError:
    gradio_major = 0
if gradio_major != 6:
    subprocess.run([sys.executable, "-m", "pip", "install", "gradio>=6,<7"], check=True)
import gradio as gr

def generate_asset(photo, subject, quality, seed_text, retry_run, progress=gr.Progress()):
    retry_run = (retry_run or "").strip()
    previous = None
    if retry_run:
        if not re.fullmatch(r"[0-9a-f]{12}", retry_run):
            raise gr.Error("Enter the 12-character run ID shown in Diagnostics.")
        previous = OUTPUT / retry_run
        if not all((previous / f"{name.lower()}.png").is_file() for name in ANGLES):
            raise gr.Error("This run has no complete set of views in the current Kaggle session.")
    if photo is None and previous is None:
        raise gr.Error("Upload one photo.")
    seed_text = (seed_text or "").strip()
    if seed_text and (len(seed_text) > 10 or not seed_text.isascii()
                      or not seed_text.isdecimal() or int(seed_text) >= 2**32):
        raise gr.Error("Use a seed from 0 to 4294967295, or leave it blank.")
    seed = int(seed_text) if seed_text else secrets.randbits(32)
    side, view_steps, mesh_steps, resolution = PRESETS[quality]
    subject = (subject or "").strip() or "the main object"
    devices = gpu_devices()
    if not devices:
        raise gr.Error("Enable a GPU in Kaggle Notebook Settings and run the cell again.")
    worker_count = min(len(devices), 2) if PARALLEL_QWEN else 1
    mesh_device = devices[1] if len(devices) > 1 else devices[0]

    directory = OUTPUT / uuid.uuid4().hex[:12]
    directory.mkdir(parents=True)
    gallery = []
    start = time.monotonic()
    download_process = None
    stage = "Qwen setup"
    yield [], None, gr.update(visible=False, value=None), "Preparing Qwen. The first setup downloads model weights.", None, ""
    try:
        progress(0.06, desc="Setting up Qwen-Image-2.1")
        if previous is None:
            prepare_comfy()
        stage = "Starting Hunyuan weight download"
        download_process = start_hunyuan_download(directory)
        if previous is not None:
            stage = "Restoring generated views"
            for name in ANGLES:
                filename = f"{name.lower()}.png"
                target = directory / filename
                shutil.copyfile(previous / filename, target)
                gallery.append((str(target), name))
            yield gallery, None, gr.update(visible=False), "Reusing the four generated views.", None, ""
        else:
            stage = "Qwen view generation"
            with ExitStack() as workers:
                process, front_api = start_comfy(devices[0], 0, directory)
                workers.callback(stop_comfy, process)
                ref = prepare_reference(photo, side, aspect=1.0)

                def render_angle(name, api, front_ref):
                    path = directory / f"{name.lower()}.png"
                    render_view(api, view_graph(ref, front_ref, subject, ANGLES[name], view_steps, seed), path)
                    return str(path), name

                progress(0.16, desc="Generating the reference front view")
                gallery.append(render_angle("Front", front_api, None))
                yield list(gallery), None, gr.update(visible=False), "Front view ready. Hunyuan weights are downloading in the background.", None, ""
                with Image.open(gallery[0][0]) as image:
                    front = prepare_reference(image, side, aspect=1.0)

                progress(0.3, desc=f"Generating remaining angles with {worker_count} Qwen worker(s)")
                if worker_count == 2:
                    process, second_api = start_comfy(devices[1], 1, directory)
                    workers.callback(stop_comfy, process)

                    def render_second_worker():
                        render_angle("Back", second_api, front)

                    # Give each worker its own queue so GPU 1 can render the back
                    # while GPU 0 renders the two profiles.
                    with ThreadPoolExecutor(max_workers=2) as pool:
                        back = pool.submit(render_second_worker)
                        profiles = pool.submit(lambda: [
                            render_angle("Left", front_api, front),
                            render_angle("Right", front_api, front),
                        ])
                        for future in as_completed((back, profiles)):
                            future.result()
                            gallery = [(str(directory / f"{name.lower()}.png"), name)
                                       for name in ANGLES if (directory / f"{name.lower()}.png").exists()]
                            yield list(gallery), None, gr.update(visible=False), f"Generated {len(gallery)} of 4 views.", None, ""
                else:
                    for name in ("Left", "Back", "Right"):
                        gallery.append(render_angle(name, front_api, front))
                        yield list(gallery), None, gr.update(visible=False), f"Generated {len(gallery)} of 4 views.", None, ""

        stage = "Hunyuan weight download"
        if download_process.poll() is None:
            progress(0.64, desc="Waiting for Hunyuan weight download")
            yield gallery, None, gr.update(visible=False), "Four views ready. Finishing the Hunyuan download.", None, ""
        exit_code = download_process.wait(timeout=1800)
        if exit_code:
            raise RuntimeError(f"Hunyuan download worker exited with code {exit_code}.")
        stage = "Hunyuan setup or mesh inference"
        progress(0.7, desc="Reconstructing the mesh in Hunyuan3D-2mv")
        yield gallery, None, gr.update(visible=False), "Building one mesh from the four views.", None, ""
        make_mesh(directory, mesh_steps, resolution, seed, mesh_device)
        stage = "Blender ZIP export"
        asset = directory / "asset.glb"
        archive = directory / "blender_asset.zip"
        with ZipFile(archive, "w", compression=ZIP_DEFLATED) as bundle:
            bundle.write(asset, asset.name)
            for name in ANGLES:
                filename = f"{name.lower()}.png"
                bundle.write(directory / filename, f"views/{filename}")
        progress(1.0)
        elapsed = (time.monotonic() - start) / 60
        yield gallery, str(asset), gr.update(value=str(archive), visible=True), (
            f"Ready · Seed {seed} · {elapsed:.1f} min. Import asset.glb into Blender via File → Import → glTF 2.0."
        ), diagnostics(directory), ""
    except Exception as error:
        if download_process is not None:
            stop_comfy(download_process)
        (directory / "studio-error.log").write_text(
            f"Stage: {stage}\n{traceback.format_exc()}", encoding="utf-8",
        )
        log_by_stage = {
            "Hunyuan weight download": "hunyuan-download.log",
            "Hunyuan setup or mesh inference": "hunyuan-inference.log",
        }
        log = directory / log_by_stage.get(stage, "studio-error.log")
        if not log.exists() and stage == "Hunyuan setup or mesh inference":
            log = directory / "hunyuan-setup.log"
        details = log.read_text(encoding="utf-8", errors="replace") if log.exists() else ""
        if len(details) > 6000:
            details = details[:4000] + "\n[Excerpt shortened. See the full run log.]\n" + details[-2000:]
        print(f"Image to Blender failed in {stage}. Run directory: {directory}", flush=True)
        traceback.print_exc()
        yield gallery, None, gr.update(visible=False), f"Failed during **{stage}**. {error} Run ID: `{directory.name}`.", diagnostics(directory), details
    finally:
        if download_process is not None:
            stop_comfy(download_process)

CSS = """
.gradio-container {max-width: 1420px !important; margin: auto !important;}
#controls, #results {padding: 18px; border: 1px solid var(--border-color-primary);
                    border-radius: 16px; background: var(--background-fill-secondary);}
"""

with gr.Blocks(title="Image to Blender Studio") as studio:
    gr.Markdown("# Image to Blender Studio")
    with gr.Row():
        with gr.Column(scale=2, min_width=300, elem_id="controls"):
            photo = gr.Image(label="Upload one photo", type="pil", image_mode="RGBA",
                             sources=["upload"], height=275)
            subject = gr.Textbox(label="Object to reconstruct (optional)", placeholder="e.g. the green armchair",
                                 max_length=180)
            with gr.Row():
                quality = gr.Dropdown(list(PRESETS), value="Draft", label="Quality")
                seed = gr.Textbox(label="Seed", placeholder="Random")
            with gr.Accordion("Retry mesh from a previous run", open=False):
                retry = gr.Textbox(label="Run ID", placeholder="12 characters from the failed run")
                gr.Markdown("Reuse the four views from a run in this Kaggle session.")
            generate = gr.Button("Generate 3D asset", variant="primary")
            gr.Markdown("Use a photo with one clear subject, or name the object you want to isolate.")
        with gr.Column(scale=3, min_width=330, elem_id="results"):
            status = gr.Markdown("Upload a photo and click Generate 3D asset.")
            gallery = gr.Gallery(label="Generated views", columns=4, object_fit="contain",
                                 height=220, interactive=False, format="png")
            model = gr.Model3D(label="Mesh preview", height=425, interactive=False)
            download = gr.DownloadButton("Download Blender asset ZIP", visible=False)
            with gr.Accordion("Diagnostics", open=True):
                error_details = gr.Textbox(label="Error log excerpt", lines=8, interactive=False)
                diagnostic_file = gr.File(label="Full run log", interactive=False)

    generate.click(generate_asset, inputs=[photo, subject, quality, seed, retry],
                   outputs=[gallery, model, download, status, diagnostic_file, error_details], concurrency_limit=1,
                   api_visibility="private")

if __name__ == "__main__":
    password = secrets.token_urlsafe(12)
    studio.queue(max_size=4)
    _, _, share_url = studio.launch(
        share=True, inline=False, prevent_thread_lock=True, server_name="127.0.0.1",
        auth=("studio", password), css=CSS, theme=gr.themes.Soft(primary_hue="amber"),
        max_file_size="12mb", blocked_paths=[str(CACHE)],
    )
    if share_url:
        safe = escape(share_url, quote=True)
        display(HTML(f'<h3><a href="{safe}" target="_blank" rel="noopener">Open Image to Blender Studio</a></h3>'
                     f'<details><summary>Studio login</summary>Username: <code>studio</code><br>'
                     f'Password: <code>{escape(password)}</code></details>'))
    else:
        print("Gradio did not create a share link. Check the tunnel error above.")
